This notebook introduces `flow`, the SysML v2 construct for declaring item flows between parts; after running it you can model the material or signal interfaces in a structural decomposition and render an interconnection diagram.

`allocate` (Ch5 nb02) shows which part performs a function. `flow` shows what passes between parts at runtime. A `flow X.port to Y.port` statement creates a `FlowUsage` element connecting two `PartUsage` members by their item ports.

This notebook adds a `BreadHandling` assembly with a `BreadLoader` and `BreadEjector`, connected by the bread item flow. It then uses `build_interconnection_intent()` and `render_sysmld()` to produce an interconnection SVG.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")

In [ ]:
# Negative control: a flow referencing a part usage that does not exist in the assembly
# raises "unresolved reference" for the undefined dotted path.
bad_source = """
package BadFlow {
    private import ScalarValues::*;
    item def Bread;
    part def Loader { part loaf : Bread; }
    part def Assembly {
        part loader : Loader;
        flow loader.loaf to undefined_ejector.loaf;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print(f"Neg control diagnostics: {bad.diagnostics[0].message!r}")

In [ ]:
from toaster.render import build_interconnection_intent, render_sysmld
from pathlib import Path
import tempfile, os

# Build the interconnection intent for BreadHandling
intent = build_interconnection_intent(model, "ToasterDemo::BreadHandling")
print(f"Parts: {[p['name'] for p in intent['parts']]}")
print(f"Flows: {intent['flows']}")

# Render to SVG
out_path = Path(tempfile.mkdtemp()) / "bread_handling.svg"
render_sysmld(intent, out_path)
print(f"SVG written: {out_path} ({os.path.getsize(out_path)} bytes)")

The `flow` relationship in SysML v2 (A-F) is parsed and stored in OpenSysML's element graph (O-S); `build_interconnection_intent()` extracts the endpoint paths via `sysx:sourceText` and `render_sysmld()` produces an SVG showing the `loader` → `ejector` item flow (E).

Try the chapter exercise in `exercises/ch05/exercise.ipynb`: add a `CoffeeFlow` part with a `pump` and a `filter`, declare a `flow pump.water to filter.water`, build the interconnection intent, and confirm the flow endpoint paths appear correctly.